# TP3C · Experimentos

Este notebook corre los experimentos sobre el simulador de `simulador.ipynb` y deja los resultados en `resultados/`. Carga sólo las celdas del simulador, y si una corrida se corta, retoma: lo que ya está guardado no se vuelve a correr.

Se corre en dos rondas. La primera da 30 réplicas de un mes a cada alternativa, para descartar. La segunda da a las que quedan el N calculado. Con 30 réplicas el intervalo es más ancho que con N, y ese costo se acepta porque baja el cómputo de las alternativas que no van a quedar.

In [1]:
import json, multiprocessing as mp, os, time
for celda in json.load(open("simulador.ipynb"))["cells"]:
    if "modelo" in celda["metadata"].get("tags", []):
        exec("".join(celda["source"]), globals())
os.makedirs("resultados", exist_ok=True)
print("escala", P["escala"], "  factor de fin de semana", round(P["factor_finde"], 3))

escala 57.3   factor de fin de semana 1.178


## Ronda 1 · Una palanca por vez

Cada alternativa mueve una sola palanca y deja las demás como hoy. Todas usan las mismas semillas, para que cada réplica de una alternativa sea comparable con la misma réplica de hoy. La fricción se prueba con umbrales 5, 10 y 20 y reducciones 0,3 y 0,6.

El clasificador nuevo se prueba sólo con 2 moderadores, porque con uno la cola se desborda. Su comparación de referencia es la alternativa de 2 moderadores con el clasificador actual, además de hoy.

In [2]:
ALTERNATIVAS = {"hoy": dict(c=1), "c2": dict(c=2), "c3": dict(c=3)}
for u in (5, 10, 20):
    for r_ in (.3, .6):
        ALTERNATIVAS[f"friccion_u{u}_r{int(r_ * 10)}"] = dict(umbral=u, reduccion=r_)
for d_ in (.3, .5, .8):
    ALTERNATIVAS[f"degradacion_{int(d_ * 10)}"] = dict(degradar=True, d=d_)
for nombre, cl in (("clasificador_85_75", (.85, .75)), ("clasificador_95_90", (.95, .9))):
    ALTERNATIVAS[f"{nombre}_c2"] = dict(c=2, clasificador=cl)
SEMILLAS = list(range(100, 130))
DIAS, CALENTAMIENTO = 30, 2

def replica(a):
    nombre, vd, semilla, escenario = a
    t0 = time.time()
    r = RedBaires(P, vd, semilla=semilla, escenario=escenario).correr(dias=DIAS, calentamiento=CALENTAMIENTO)
    try:
        controles(r); ok = "ok"
    except AssertionError as e:
        ok = str(e)
    return dict(alternativa=nombre, semilla=semilla, **kpis(r), control=ok, segundos=time.time() - t0)

def log(archivo, texto):
    with open(archivo, "a") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + texto + "\n")

def correr_ronda(carpeta, alternativas, semillas, escenario=None):
    """Una alternativa por vez, en 4 procesos. Se guarda al terminar cada una; si ya está completa, no se vuelve a correr."""
    os.makedirs(carpeta, exist_ok=True)
    salida_log = carpeta + ".log"
    with mp.get_context("fork").Pool(4) as pool:
        for nombre, vd in alternativas.items():
            ruta = f"{carpeta}/{nombre}.csv"
            if os.path.exists(ruta) and len(pd.read_csv(ruta)) == len(semillas):
                log(salida_log, f"{nombre}: ya estaba"); continue
            t0 = time.time()
            filas = pool.map(replica, [(nombre, vd, s, escenario) for s in semillas], chunksize=1)
            pd.DataFrame(filas).to_csv(ruta + ".tmp", index=False); os.replace(ruta + ".tmp", ruta)
            malos = sum(f["control"] != "ok" for f in filas)
            log(salida_log, f"{nombre}: {len(filas)} réplicas en {time.time() - t0:.0f} s, controles fallidos: {malos}")
    log(salida_log, "ronda terminada")

In [3]:
correr_ronda("resultados/ronda1", ALTERNATIVAS, SEMILLAS)
ronda1 = pd.concat([pd.read_csv(f"resultados/ronda1/{n}.csv") for n in ALTERNATIVAS])
assert len(ronda1) == len(ALTERNATIVAS) * len(SEMILLAS), "faltan réplicas"
assert (ronda1.control == "ok").all(), ronda1.loc[ronda1.control != "ok", ["alternativa", "semilla", "control"]].to_string()
print("réplicas:", len(ronda1), " controles: ok")

réplicas: 420  controles: ok


### Resumen de la ronda 1

Costo total por mes (`costo_mes`, en miles de USD) con su intervalo de confianza del 95 %, impresiones nocivas servidas en el mes (millones), legítimas perdidas como porcentaje de las de hoy en la misma réplica (cascadas frenadas más la pérdida directa por fricción) y latencia mediana de moderación (minutos). La última columna dice si la alternativa respeta el tope del 15 % de legítimas perdidas (decisión 1).

In [4]:
import itertools, math
from scipy import stats

TOPE = 0.15                          # decisión 1: tope de legítimas perdidas

def por_replica(df, nombre, base="hoy"):
    """Costo y legítimas perdidas réplica a réplica, contra la misma réplica de hoy."""
    b = df[df.alternativa == base].set_index("semilla")
    a = df[df.alternativa == nombre].set_index("semilla").loc[b.index]
    costo = np.array([costo_mes(a.loc[s], b.loc[s], P["C"]) for s in b.index])
    perdidas = np.array([legitimas_perdidas(b.loc[s], a.loc[s]) / b.loc[s, "legitimas"] for s in b.index])
    return costo, perdidas, a

def ic(x):
    return x.mean(), stats.t.ppf(.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))

def resumen(df, nombres):
    filas = []
    for n in nombres:
        costo, perdidas, a = por_replica(df, n)
        (cm, ch), (nm, nh), (pm, ph) = ic(costo / 1e3), ic(a.nocivas.to_numpy() / 1e6), ic(perdidas * 100)
        filas.append({"alternativa": n, "costo (miles USD/mes)": f"{cm:,.0f} ± {ch:,.0f}", "nocivas (M)": f"{nm:,.0f} ± {nh:,.0f}",
                      "legítimas perdidas (%)": f"{pm:.1f} ± {ph:.1f}", "latencia mediana (min)": round(a.lat_mediana.mean(), 1),
                      "dentro del tope": pm / 100 <= TOPE})
    return pd.DataFrame(filas)

tabla1 = resumen(ronda1, list(ALTERNATIVAS))
tabla1.to_csv("resultados/ronda1_resumen.csv", index=False)
tabla1

,alternativa,costo (miles USD/mes),nocivas (M),legítimas perdidas (%),latencia mediana (min),dentro del tope
0,hoy,"5,903 ± 54",987 ± 9,0.0 ± 0.0,174.0,True
1,c2,"5,712 ± 35",949 ± 6,1.4 ± 0.2,90.5,True
2,c3,"5,728 ± 35",949 ± 6,1.4 ± 0.2,90.4,True
3,friccion_u5_r3,"5,027 ± 27",773 ± 5,29.3 ± 0.2,91.7,False
4,friccion_u5_r6,"4,503 ± 22",662 ± 4,39.9 ± 0.2,90.9,False
5,friccion_u10_r3,"5,202 ± 30",815 ± 5,23.7 ± 0.2,92.4,False
6,friccion_u10_r6,"4,792 ± 25",727 ± 4,32.2 ± 0.2,91.3,False
7,friccion_u20_r3,"5,557 ± 35",901 ± 6,12.2 ± 0.2,98.4,True
8,friccion_u20_r6,"5,350 ± 32",856 ± 5,16.5 ± 0.2,94.1,False
9,degradacion_3,"5,575 ± 33",840 ± 5,14.0 ± 0.2,95.6,True


## Ronda 2 · Finalistas y combinaciones

**Finalistas.** Pasan las alternativas que respetan el tope del 15 %: 2 moderadores, fricción con umbral 20 y reducción 0,3, degradación 0,3 y los dos clasificadores con 2 moderadores.

**N.** Se calcula sobre la diferencia apareada de costo entre finalistas, con el error admitido de la parte B: ±7.920 USD por mes, medio sueldo de moderador. La decisión se toma entre finalistas; contra hoy las diferencias son de cientos de miles de USD y no hacen falta tantas réplicas para verlas. Se toma el N del par más exigente.

**Combinaciones (decisión 9).** Cada palanca entra prendida, con la intensidad finalista, o apagada, como hoy: 2⁴ = 16 combinaciones, menos las 4 que tienen el clasificador nuevo con un solo moderador, porque con uno la cola se desborda. Quedan 12. El clasificador entra con 95/90, el más barato de los dos. Las combinaciones que ya se corrieron en la ronda 1 sólo suman las semillas que faltan: como todas usan las mismas semillas, siguen apareadas.

In [5]:
FINALISTAS = ["c2", "friccion_u20_r3", "degradacion_3", "clasificador_85_75_c2", "clasificador_95_90_c2"]
ERROR = 7920                         # USD por mes, el mismo error admitido de la parte B

costos1 = {n: por_replica(ronda1, n)[0] for n in FINALISTAS}
t30 = stats.t.ppf(.975, len(SEMILLAS) - 1)
pares = pd.DataFrame([{"a": a, "b": b, "s_dif": (costos1[a] - costos1[b]).std(ddof=1)} for a, b in itertools.combinations(FINALISTAS, 2)])
pares["N"] = np.ceil((t30 * pares.s_dif / ERROR) ** 2).astype(int)
N = max(int(pares.N.max()), len(SEMILLAS))
SEMILLAS2 = list(range(100, 100 + N))
print("N =", N, "réplicas por combinación")
pares.sort_values("N", ascending=False).round(0)

N = 73 réplicas por combinación


,a,b,s_dif,N
8,degradacion_3,clasificador_95_90_c2,32947.0,73
7,degradacion_3,clasificador_85_75_c2,32549.0,71
4,friccion_u20_r3,degradacion_3,31346.0,66
3,c2,clasificador_95_90_c2,26382.0,47
6,friccion_u20_r3,clasificador_95_90_c2,25866.0,45
1,c2,degradacion_3,25066.0,42
5,friccion_u20_r3,clasificador_85_75_c2,22573.0,34
2,c2,clasificador_85_75_c2,22443.0,34
0,c2,friccion_u20_r3,21249.0,31
9,clasificador_85_75_c2,clasificador_95_90_c2,12410.0,11


In [6]:
PALANCAS = {"c2": dict(c=2), "fr": dict(umbral=20, reduccion=.3), "dg": dict(degradar=True, d=.3), "cl": dict(clasificador=(.95, .9))}
COMBOS = {}
for prendidas in itertools.product((0, 1), repeat=len(PALANCAS)):
    nombres = [p for p, on in zip(PALANCAS, prendidas) if on]
    if "cl" in nombres and "c2" not in nombres:
        continue                     # clasificador nuevo con un solo moderador: la cola se desborda
    vd = dict(c=1)
    for p in nombres:
        vd.update(PALANCAS[p])
    COMBOS["+".join(nombres) or "hoy"] = vd
assert len(COMBOS) == 12

# Las que ya están en la ronda 1, con el nombre que tienen ahí.
DE_RONDA1 = {"hoy": "hoy", "c2": "c2", "fr": "friccion_u20_r3", "dg": "degradacion_3", "c2+cl": "clasificador_95_90_c2"}
for combo, viejo in DE_RONDA1.items():
    assert COMBOS[combo] == {"c": 1, **ALTERNATIVAS[viejo]} or COMBOS[combo] == ALTERNATIVAS[viejo], combo

correr_ronda("resultados/ronda2", {k: v for k, v in COMBOS.items() if k in DE_RONDA1}, SEMILLAS2[len(SEMILLAS):])
correr_ronda("resultados/ronda2", {k: v for k, v in COMBOS.items() if k not in DE_RONDA1}, SEMILLAS2)

partes = []
for combo in COMBOS:
    r2 = pd.read_csv(f"resultados/ronda2/{combo}.csv").assign(alternativa=combo)
    if combo in DE_RONDA1:
        r2 = pd.concat([ronda1[ronda1.alternativa == DE_RONDA1[combo]].assign(alternativa=combo), r2])
    partes.append(r2)
ronda2 = pd.concat(partes, ignore_index=True)
assert (ronda2.groupby("alternativa").semilla.nunique() == N).all(), "faltan réplicas"
assert (ronda2.control == "ok").all(), ronda2.loc[ronda2.control != "ok", ["alternativa", "semilla", "control"]].to_string()
print("réplicas:", len(ronda2), " controles: ok")

réplicas: 876  controles: ok


### Resumen de la ronda 2

Mismas columnas que la ronda 1, con N réplicas por combinación.

In [7]:
tabla2 = resumen(ronda2, list(COMBOS))
tabla2.to_csv("resultados/ronda2_resumen.csv", index=False)
tabla2

,alternativa,costo (miles USD/mes),nocivas (M),legítimas perdidas (%),latencia mediana (min),dentro del tope
0,hoy,"5,885 ± 30",984 ± 5,0.0 ± 0.0,159.3,True
1,dg,"5,572 ± 20",840 ± 3,13.9 ± 0.1,95.6,True
2,fr,"5,554 ± 21",901 ± 3,12.1 ± 0.1,98.1,True
3,fr+dg,"5,397 ± 19",791 ± 3,22.8 ± 0.1,92.8,False
4,c2,"5,711 ± 22",949 ± 4,1.3 ± 0.1,90.5,True
5,c2+cl,"5,193 ± 20",870 ± 3,-1.8 ± 0.1,92.0,True
6,c2+dg,"5,548 ± 20",833 ± 3,14.2 ± 0.1,90.5,True
7,c2+dg+cl,"5,114 ± 18",766 ± 3,11.7 ± 0.1,91.3,True
8,c2+fr,"5,526 ± 20",893 ± 3,12.4 ± 0.1,90.5,True
9,c2+fr+cl,"5,135 ± 19",831 ± 3,10.7 ± 0.1,91.5,True
